# Tutorial1
We show how to train a profitable RL agent on portfolio managemnt with EIIE algorithm on US stock market.

## Step 1: Import Packages
Modify the system path and load the corresponding packages and functions

In [ ]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
ROOT = os.path.dirname(os.path.abspath("."))
sys.path.append(ROOT)
import torch
import argparse
import os.path as osp
from mmcv import Config
from trademaster.utils import replace_cfg_vals
from trademaster.nets.builder import build_net
from trademaster.environments.builder import build_environment
from trademaster.datasets.builder import build_dataset
from trademaster.agents.builder import build_agent
from trademaster.optimizers.builder import build_optimizer
from trademaster.losses.builder import build_loss
from trademaster.trainers.builder import build_trainer
from trademaster.transition.builder import build_transition
from trademaster.utils import plot
from trademaster.utils import set_seed
set_seed(2023)

2025-11-01 13:46:20,251	ERROR services.py:1360 -- Failed to start the dashboard , return code 1
2025-11-01 13:46:20,251	ERROR services.py:1385 -- Error should be written to 'dashboard.log' or 'dashboard.err'. We are printing the last 20 lines for you. See 'https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#logging-directory-structure' to find where the log file is.
2025-11-01 13:46:20,251	ERROR services.py:1429 -- 
The last 20 lines of C:\Users\zwd_1\AppData\Local\Temp\ray\session_2025-11-01_13-46-17_731260_16488\logs\dashboard.log (it contains the error message from the dashboard): 
    from ray.util.state.api import (
  File "e:\work\condaenvs\trademaster\Lib\site-packages\ray\util\state\api.py", line 18, in <module>
    from ray.util.state.common import (
  File "e:\work\condaenvs\trademaster\Lib\site-packages\ray\util\state\common.py", line 1001, in <module>
    @dataclass(init=not IS_PYDANTIC_2)
     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\work

## Step 2: Load Configs
Load default config from the file `configs/portfolio_management/portfolio_management_dj30_eiie_eiie_adam_mse.py`

In [ ]:
parser = argparse.ArgumentParser(description='Download Alpaca Datasets')
parser.add_argument("--config", default=osp.join(ROOT, "configs", "portfolio_management", "portfolio_management_dj30_eiie_eiie_adam_mse.py"),
                    help="download datasets config file path")
parser.add_argument("--task_name", type=str, default="train")

args, _= parser.parse_known_args()
cfg = Config.fromfile(args.config)
task_name = args.task_name
cfg = replace_cfg_vals(cfg)

In [ ]:
from pprint import pprint
pprint(cfg)

## Step 3: Build Dataset

In [ ]:
dataset = build_dataset(cfg)

## Step 4: Build Reinforcement Learning Environments

In [ ]:
train_environment = build_environment(cfg, default_args=dict(dataset=dataset, task="train"))
valid_environment = build_environment(cfg, default_args=dict(dataset=dataset, task="valid"))
test_environment = build_environment(cfg, default_args=dict(dataset=dataset, task="test"))

In [ ]:
train_environment.df.head()

## Step 5: Build Net 
Update information on the state and action dimension. Crreate networks and optimizer for EIIE.

In [ ]:
action_dim = train_environment.action_dim # 29
state_dim = train_environment.state_dim # 11
input_dim = len(train_environment.tech_indicator_list)
time_steps = train_environment.time_steps

cfg.act.update(dict(input_dim=input_dim, time_steps=time_steps))
cfg.cri.update(dict(input_dim=input_dim, action_dim= action_dim, time_steps=time_steps))

act = build_net(cfg.act)
cri = build_net(cfg.cri)
act_optimizer = build_optimizer(cfg, default_args=dict(params=act.parameters()))
cri_optimizer = build_optimizer(cfg, default_args=dict(params=cri.parameters()))

## Step 6: Build Loss Function

In [ ]:
criterion = build_loss(cfg)

## Step 7: Build Transition

In [ ]:
transition = build_transition(cfg)


## Step 8: Build Agent

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
agent = build_agent(cfg, default_args=dict(action_dim=action_dim,state_dim=state_dim,time_steps = time_steps,act=act,cri=cri,act_optimizer=act_optimizer,cri_optimizer = cri_optimizer,criterion=criterion,transition = transition,device = device))

## Step 9: Build Trainer
Build trainer from config and create work directionary to save the result, model and config

In [ ]:
trainer = build_trainer(cfg, default_args=dict(train_environment=train_environment,valid_environment=valid_environment,test_environment=test_environment,agent=agent,device=device))
work_dir = os.path.join(ROOT, cfg.trainer.work_dir)

if not os.path.exists(work_dir):
    os.makedirs(work_dir)
cfg.dump(osp.join(work_dir, osp.basename(args.config)))

## Step 10: RL Agent Training
Train the EIIE agent based on the config and save results in workdir

In [ ]:
trainer.train_and_valid()

## Step 11: RL Agent Testing

In [ ]:
trainer.test();

In [ ]:
plot(trainer.test_environment.save_asset_memory(),alg="EIIE")